# ProteinBERT fluorescence regression

**Zirui Chen · Deep learning coursework, HW9 · Portfolio edition**

Fine-tune pretrained ProteinBERT to predict a continuous fluorescence target from an amino-acid sequence. The assignment names the [official signal-peptide demo](https://github.com/nadavbra/protein_bert/blob/master/ProteinBERT%20demo.ipynb) as the adaptation template.

This notebook reorganizes the submitted code without running it. Archived test results: **R² 0.6978, RMSE 0.5355, MAE 0.3868, Pearson r 0.8686** on 27,217 recorded test rows. These are not independently reproduced results.

![Original observed-versus-predicted plot](../figures/predicted_vs_observed_original.png)

The task is regression. No confusion matrix or discrete class definition is present in the original implementation. See [method notes](../docs/method_notes.md) for this distinction and the discrepancy between source learning rates and the saved log.

## 1. Environment and local paths

Review [environment limitations](../docs/method_notes.md) before execution. Original package versions are unknown. Reading the saved result does not require running the notebook. Executing all cells starts fine-tuning and requires the two CSVs and pretrained checkpoint.

In [ ]:
from pathlib import Path
ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
if not (ROOT / 'data' / 'README.md').exists():
    raise RuntimeError('Open from the repository root or notebooks directory.')
DATA_DIR = ROOT / 'data' / 'raw'
MODEL_DIR = ROOT / 'models'


In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display
import tensorflow as tf
from tensorflow import keras
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split
from proteinbert import OutputType, OutputSpec, FinetuningModelGenerator, load_pretrained_model, finetune
from proteinbert.conv_and_global_attention_model import get_model_with_hidden_layers_as_outputs
from proteinbert import model_generation


## 2. Original optimizer compatibility patch

This modifies Adam globally in the current kernel. Its placeholder optimizer-state methods are retained as original implementation details, not a verified compatibility solution.

In [ ]:
if not getattr(tf.keras.optimizers.Adam, '_proteinbert_compat_patched', False):
    orig_adam_init = tf.keras.optimizers.Adam.__init__

    def patched_adam_init(self, *args, **kwargs):
        if 'lr' in kwargs:
            kwargs['learning_rate'] = kwargs.pop('lr')
        orig_adam_init(self, *args, **kwargs)
        if not hasattr(self, 'amsgrad'):
            self.amsgrad = False
        if not hasattr(self, 'beta_1'):
            self.beta_1 = getattr(self, 'beta_1', 0.9)
        if not hasattr(self, 'beta_2'):
            self.beta_2 = getattr(self, 'beta_2', 0.999)
        if not hasattr(self, 'epsilon'):
            self.epsilon = getattr(self, 'epsilon', 1e-07)
        if not hasattr(self, 'get_weights'):
            self._proteinbert_weights = None
            self.get_weights = lambda: self._proteinbert_weights if self._proteinbert_weights is not None else []
            self.set_weights = lambda w: setattr(self, '_proteinbert_weights', w)
    tf.keras.optimizers.Adam.__init__ = patched_adam_init
    tf.keras.optimizers.Adam._proteinbert_compat_patched = True
    print('Applied the original coursework Adam compatibility patch; see method notes.')


## 3. Load and clean the data

Required columns: `seq` and `fluorescence`. Missing-value and duplicate filtering apply to whole rows. The recorded cleaned counts are 21,446 training-file rows and 27,217 test-file rows. See [data specification](../data/README.md).

In [ ]:
train_set = pd.read_csv(DATA_DIR / 'fluorescence.train.csv').dropna().drop_duplicates()
test_set = pd.read_csv(DATA_DIR / 'fluorescence.test.csv').dropna().drop_duplicates()
print(f'{len(train_set)} training records, {len(test_set)} test records.')
print('\nFluorescence intensity statistics:')
print(f"Training set - Mean: {train_set['fluorescence'].mean():.4f}, Std: {train_set['fluorescence'].std():.4f}")
print(f"Test set - Mean: {test_set['fluorescence'].mean():.4f}, Std: {test_set['fluorescence'].std():.4f}")


## 4. Configure a regression output

`OutputType(False, 'numeric')` defines a sequence-level continuous target. No classification thresholds are introduced.

In [ ]:
OUTPUT_TYPE = OutputType(False, 'numeric')
OUTPUT_SPEC = OutputSpec(OUTPUT_TYPE, None)
print("Perform fluorescence intensity prediction using OutputType(False, 'numeric')")


## 5. Load pretrained ProteinBERT

Use the original local `epoch_92400_sample_23500000.pkl` checkpoint. Automatic downloading remains disabled. Check [model instructions](../models/README.md).

In [ ]:
pretrained_model_generator, input_encoder = load_pretrained_model(local_model_dump_dir=str(MODEL_DIR), local_model_dump_file_name='epoch_92400_sample_23500000.pkl', download_model_dump_if_not_exists=False)


## 6. Model adaptation and validation split

Expose hidden-layer representations through the upstream helper, apply dropout 0.5 and reserve 10% of the training file with split seed 42. The original code does not set a complete training random seed.

In [ ]:
model_generator = FinetuningModelGenerator(pretrained_model_generator, OUTPUT_SPEC, pretraining_model_manipulation_function=get_model_with_hidden_layers_as_outputs, dropout_rate=0.5)
train_seqs, val_seqs, train_labels, val_labels = train_test_split(train_set['seq'], train_set['fluorescence'], test_size=0.1, random_state=42)
training_callbacks = [keras.callbacks.ReduceLROnPlateau(patience=1, factor=0.25, min_lr=1e-05, verbose=1), keras.callbacks.EarlyStopping(patience=2, restore_best_weights=True)]


## 7. Staged fine-tuning

This cell starts training. The frozen, full-model and final long-sequence stages are preserved. **Source learning-rate arguments differ from the saved log**, whose initial value is 2e-4 in all three stages. Do not treat these source arguments as confirmed effective settings.

In [ ]:
finetune(model_generator, input_encoder, OUTPUT_SPEC, train_seqs, train_labels, val_seqs, val_labels, seq_len=512, batch_size=32, max_epochs_per_stage=20, lr=0.0001, begin_with_frozen_pretrained_layers=True, lr_with_frozen_pretrained_layers=0.01, n_final_epochs=1, final_seq_len=1024, final_lr=1e-05, callbacks=training_callbacks)


## 8. Regression evaluation helper

MSE, RMSE, MAE and R² assess continuous prediction errors; `numpy.corrcoef` gives Pearson correlation. Evaluation uses sequence length 512.

In [ ]:
def evaluate_regression(model_generator, input_encoder, output_spec, sequences, true_values, seq_len=512, batch_size=32):
    model = model_generator.create_model(seq_len)
    encoded_x = input_encoder.encode_X(sequences, seq_len)
    predictions = model.predict(encoded_x, batch_size=batch_size)
    if isinstance(predictions, list):
        predictions = predictions[0]
    predictions = predictions.flatten()
    true_values = np.array(true_values)
    mse = mean_squared_error(true_values, predictions)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(true_values, predictions)
    r2 = r2_score(true_values, predictions)
    correlation = np.corrcoef(true_values, predictions)[0, 1]
    return {'MSE': mse, 'RMSE': rmse, 'MAE': mae, 'R2_Score': r2, 'Correlation': correlation, 'Predictions': predictions, 'True_Values': true_values}


## 9. Test evaluation and scatter plot

This block requires the in-memory fine-tuned model. Its newly computed output, if run later, must not be assumed identical to the archived result shown above. The source displays only the first ten predictions and does not save the full prediction array.

In [ ]:
print('\nEvaluating on test set...')
test_results = evaluate_regression(model_generator, input_encoder, OUTPUT_SPEC, test_set['seq'], test_set['fluorescence'], seq_len=512, batch_size=32)
print('\nTest-set performance:')
print(f"MSE: {test_results['MSE']:.4f}")
print(f"RMSE: {test_results['RMSE']:.4f}")
print(f"MAE: {test_results['MAE']:.4f}")
print(f"R² Score: {test_results['R2_Score']:.4f}")
print(f"Correlation: {test_results['Correlation']:.4f}")
results_df = pd.DataFrame({'Sequence': test_set['seq'], 'True_Fluorescence': test_results['True_Values'], 'Predicted_Fluorescence': test_results['Predictions']})
display(results_df.head(10))
plt.figure(figsize=(8, 6))
plt.scatter(test_results['True_Values'], test_results['Predictions'], alpha=0.6)
plt.plot([test_results['True_Values'].min(), test_results['True_Values'].max()], [test_results['True_Values'].min(), test_results['True_Values'].max()], 'r--', lw=2)
plt.xlabel('True Fluorescence Intensity')
plt.ylabel('Predicted Fluorescence Intensity')
plt.title(f"Fluorescence Intensity Prediction\nR² = {test_results['R2_Score']:.4f}")
plt.show()


## Interpretation

The archived run shows a positive association between observed and predicted fluorescence (Pearson r 0.8686), with R² 0.6978. R² must not be described as 69.78% classification accuracy. The scatter plot and error metrics are the appropriate recorded outputs for this regression task.

No full prediction file, fine-tuned checkpoint, exact environment or verified CSV lineage accompanies the submission. Learning-rate inconsistency, sequence independence and the role of the final fine-tuning stage remain reproducibility questions. See [method notes](../docs/method_notes.md) and [recorded outputs](../results/README.md).

## Reference

Brandes N., Ofer D., Peleg Y., Rappoport N., and Linial M. (2022). [ProteinBERT: a universal deep-learning model of protein sequence and function](https://doi.org/10.1093/bioinformatics/btac020). Bioinformatics, 38(8), 2102–2110. [Official implementation](https://github.com/nadavbra/protein_bert).